# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` values as per the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Croissant Schema URL:** [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant -U

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title: {}".format(metadata.name))
print("Description: {}".format(metadata.description))

## 2. Data Overview
Review available record sets and their schema. 

All entities are referenced by their `@id` as per the Croissant specification.

We enumerate all `RecordSet`s and their fields, reporting their `@id`.

In [ ]:
# List all RecordSets in the dataset
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets were enumerated in the top-level metadata. If this is unexpected, the Croissant file may need to be updated.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  name: {rs.get('name', '')}")
        print(f"  description: {rs.get('description', '')}")
        print("  Fields/columns:")
        for field in rs.get('field', []):
            print(f"    Field @id: {field['@id']} | name: {field.get('name', '')} | dataType: {field.get('dataType', '')}")

Next, we'll load the first (or only) record set and show example records and field names. **You must use the `@id` of the record set** as found above. 
If there are multiple record sets, adjust the list to include the desired ones.

In [ ]:
# Discover all RecordSet @ids from metadata for loading records
# We will list the first few records from each RecordSet by @id
record_sets_with_ids = list(dataset.record_sets)
record_set_ids = [rs['@id'] for rs in record_sets_with_ids]
pprint({f"record_set_{i}": rid for i, rid in enumerate(record_set_ids)})

for record_set_id in record_set_ids:
    print(f"\nSample records for RecordSet @id: {record_set_id}")
    try:
        # Show the first 3 records
        for i, rec in enumerate(dataset.records(record_set=record_set_id)):
            print(rec)
            if i >= 2:
                break
    except Exception as exc:
        print(f"Failed to fetch records for {record_set_id}: {exc}")

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for further analysis.

Replace `<record_set_id>` below with one of the discovered `@id` values. All references to individual fields (columns) will use their unique `@id` as well, following best practice.

In [ ]:
# Example: Extract data from all available record sets into DataFrames, referenced by @id
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Columns for RecordSet @id {record_set_id}:\n {df.columns.tolist()}")
            print(df.head(), "\n")
    except Exception as exc:
        print(f"Could not load data for RecordSet {record_set_id}: {exc}")

## 4. Exploratory Data Analysis (EDA)
Now let's select a numeric field in a particular record set—referenced by its `@id`—and demonstrate common analysis steps, such as filtering, normalization, and grouping.

In [ ]:
# Choose one record set and one or more numeric fields, using their @id
# You'll need to adjust the record_set_id and numeric_field to those output above.

# For example, suppose our main RecordSet @id is 'https://api.app.sen.science/frontiers/7154140/ev_proteins' and
# it has a numeric field '@id' 'coverage_percentage'.

chosen_record_set_id = record_set_ids[0] if record_set_ids else None
if not chosen_record_set_id:
    print("No record sets discovered in the schema.")
else:
    df = dataframes[chosen_record_set_id]
    print(f"Operating on record set @id: {chosen_record_set_id}")
    
    # Heuristically pick a numeric field, or let user pick. List all columns and try floats/ints.
    num_cols = df.select_dtypes(include=['int', 'float']).columns.tolist()
    if not num_cols:
        # Attempt coercing object columns
        numeric_candidates = []
        for col in df.columns:
            # try to convert to numeric
            try:
                vals = pd.to_numeric(df[col], errors='coerce')
                if vals.notnull().sum() > (0.6*len(df)):
                    numeric_candidates.append(col)
            except Exception:
                continue
        num_cols = numeric_candidates
    if not num_cols:
        print("No usable numeric fields detected in this record set.")
    else:
        numeric_field = num_cols[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10

        # Filter records above the mean (as generic threshold)
        filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize (z-score) this numeric field
        filtered_df[numeric_field + '_normalized'] = (
            pd.to_numeric(filtered_df[numeric_field], errors='coerce') -
            pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()
        ) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()

        print(f"Normalized {numeric_field} (z-score):")
        print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Group by a likely categorical field
        # Try 'description', 'accession', or another field as group field
        possible_groups = [c for c in df.columns if ('desc' in c.lower() or 'sample' in c.lower() or 'group' in c.lower())]
        group_field = possible_groups[0] if possible_groups else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            print(grouped_df.head())

## 5. Visualization
Plot data distributions or relationships for one or two fields, referencing them by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set_id and num_cols:
    # Histogram of the numeric field
    plt.figure(figsize=(6,4))
    pd.to_numeric(df[numeric_field], errors='coerce').hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field} in @id {chosen_record_set_id}')
    plt.show()

    # Boxplot by group_field, if available
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
- We loaded the FAIR² mass spec dataset using `mlcroissant` directly from its Croissant schema URL, referencing all fields and record sets by their `@id`.
- Initial exploration reveals the available record sets and their columns. We've demonstrated how to process, filter, group, and visualize data directly referencing schema identifiers.
- This workflow enables reproducible FAIR data analysis across any Croissant-compliant dataset.